In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [2]:
from concurrent.futures import ThreadPoolExecutor

from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

NUM_TRAINING_TOKENS = int(1e8)

def load_saes(checkpoint_dir: str):
    saes = {}

    def load_layer_checkpoint(layer):
        checkpoint = find_latest_checkpoint(checkpoint_dir, layer)
        if checkpoint is not None:
            cp = load_checkpoint(checkpoint)
            if cp.total_tokens_trained < NUM_TRAINING_TOKENS:
                print(f"No checkpoint found for layer {layer}")
                return layer, None
            sae = load_checkpoint(checkpoint).sae
            sae.eval()
            sae.onload()
            print(f"Loaded checkpoint for layer {layer}")
            return layer, sae
        else:
            print(f"No checkpoint found for layer {layer}")
            return layer, None

    # Load the latest checkpoints for each layer in parallel
    with ThreadPoolExecutor() as executor:
        results = executor.map(
            load_layer_checkpoint, range(model.num_layers - 1, -1, -1)
        )
        for layer, sae in results:
            if sae is not None:
                saes[layer] = sae

    return saes

saes = load_saes("/workspace/sae_checkpoints/gemma_2_2b/next_layer_finetuned_interaction")
# saes = load_saes("/workspace/sae_checkpoints/gemma_2_2b/next_layer_finetuned")

Loaded checkpoint for layer 1
Loaded checkpoint for layer 20
Loaded checkpoint for layer 21
Loaded checkpoint for layer 11
Loaded checkpoint for layer 25
Loaded checkpoint for layer 15
Loaded checkpoint for layer 19
Loaded checkpoint for layer 7
Loaded checkpoint for layer 17
Loaded checkpoint for layer 24
Loaded checkpoint for layer 3
Loaded checkpoint for layer 22
Loaded checkpoint for layer 23
Loaded checkpoint for layer 5
Loaded checkpoint for layer 0
Loaded checkpoint for layer 10
Loaded checkpoint for layer 13
Loaded checkpoint for layer 2
Loaded checkpoint for layer 16
Loaded checkpoint for layer 4
Loaded checkpoint for layer 6
Loaded checkpoint for layer 9
Loaded checkpoint for layer 18
Loaded checkpoint for layer 14
Loaded checkpoint for layer 8
Loaded checkpoint for layer 12


In [3]:
import cloudpickle

replacement_thresholds = cloudpickle.load(
    open(
        "/workspace/sae_checkpoints/validations/gemma_2_2b/next_layer_finetuned_interaction/0.activation_thresholds",
        # "/workspace/sae_checkpoints/validations/gemma_2_2b/next_layer_finetuned/0.activation_thresholds",
        "rb",
    )
)
for layer, sae in saes.items():
    for i, a in enumerate(sae.encoder.activation):
        a.threshold.fill_(replacement_thresholds[layer][i])


In [4]:
from transformers_sae.ops import generate
from transformers_sae.replacement_model import make_replacement_model

with torch.inference_mode():
    generate(
        ["The capital of France,",],
        make_replacement_model(model, saes),
        tokenizer,
        stream=True,
    )


 the capital of the country, is a great place to visit. The city is a great place to


In [76]:
list(MMLUTask)[:10]

[<MMLUTask.HIGH_SCHOOL_EUROPEAN_HISTORY: 'high_school_european_history'>,
 <MMLUTask.BUSINESS_ETHICS: 'business_ethics'>,
 <MMLUTask.CLINICAL_KNOWLEDGE: 'clinical_knowledge'>,
 <MMLUTask.MEDICAL_GENETICS: 'medical_genetics'>,
 <MMLUTask.HIGH_SCHOOL_US_HISTORY: 'high_school_us_history'>,
 <MMLUTask.HIGH_SCHOOL_PHYSICS: 'high_school_physics'>,
 <MMLUTask.HIGH_SCHOOL_WORLD_HISTORY: 'high_school_world_history'>,
 <MMLUTask.VIROLOGY: 'virology'>,
 <MMLUTask.HIGH_SCHOOL_MICROECONOMICS: 'high_school_microeconomics'>,
 <MMLUTask.ECONOMETRICS: 'econometrics'>]

In [22]:
from transformers_sae.benchmark import BenchmarkModel, MMLUBenchmark, MMLUTask
import cloudpickle

# mmlu_baseline = run_mmlu(model, tokenizer, 16, tasks=[MMLUTask.HIGH_SCHOOL_BIOLOGY])
baseline_benchmark_model = BenchmarkModel(model, tokenizer)
mmlu_baseline = MMLUBenchmark(tasks=[MMLUTask.HIGH_SCHOOL_BIOLOGY])
mmlu_baseline.evaluate(model=baseline_benchmark_model, batch_size=16)
# cloudpickle.dump(mmlu_baseline, open("/workspace/sae_checkpoints/benchmarks/baseline", "wb"))

RuntimeError: super(): __class__ cell not found

In [10]:
from transformers_sae.benchmark import BenchmarkModel, MMLUBenchmark, MMLUTask
import cloudpickle

sae_benchmark_model = BenchmarkModel(make_replacement_model(model, saes), tokenizer)
# mmlu_sae = MMLUBenchmark(tasks=list(MMLUTask)[:2])
mmlu_sae = MMLUBenchmark(n_problems_per_task=16, tasks=[*list(MMLUTask)[:1], MMLUTask.HIGH_SCHOOL_BIOLOGY])
mmlu_sae.evaluate(model=sae_benchmark_model, batch_size=16);

Batch Processing high_school_european_history (batch_size=16):   0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([16, 2966])
MMLU Task Accuracy (task=high_school_european_history): 0.0


Batch Processing high_school_biology (batch_size=16):   0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([16, 557])
MMLU Task Accuracy (task=high_school_biology): 0.1875
Overall MMLU Accuracy: 0.09375


Traceback (most recent call last):
  File "/cloud-dev/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 533, in _process_child_correspondence
    orig.apply_correspondence(
    ~~~~~~~~~~~~~~~~~~~~~~~~~^
        ccorr,
        ^^^^^^
        order=order,
        ^^^^^^^^^^^^
        controller=controller,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/cloud-dev/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 747, in apply_correspondence
    self.recode(new_code, recode_current=False)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/cloud-dev/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 735, in recode
    conform(co, subcode, use_cache=use_cache)
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py", line 144, in conform
    fv1 = obj1.__code__.co_freevars
          ^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute '__code__'. Did you mean: '__

In [78]:
cloudpickle.dump(mmlu_sae, open("/workspace/sae_checkpoints/benchmarks/next_layer_finetuned_interaction", "wb"))

In [82]:
from collections import Counter
print(Counter(mmlu_sae.predictions["Prediction"]))
print(Counter(mmlu_baseline.predictions["Prediction"]))

Counter({'A': 272, 'B': 24, 'C': 8, 'D': 5, 'S': 1})
Counter({'C': 4332, 'B': 3925, 'A': 3266, 'D': 2514, '': 5})
